# 07 · Clustering from scratch + the pregnancy changes

Following the tutorial notebook [`07_clustering_scratch_and_differential_student.ipynb`](https://github.com/lamanno-epfl/lipidomics_tutorial_cajalcourse/blob/main/notebooks/level3/07_clustering_scratch_and_differential_student.ipynb).

**Kernel:** `cajal-lipidomics`

**What I'm doing here**
- Mann-Whitney U, log2 fold change, Benjamini-Hochberg, built by hand
- A divisive splitter gated on differential lipids
- Pregnant vs control on the uMAIA layer: overall, volcano, per lipizone
- Composite scores: membrane remodeling and myelination

**My notes / what surprised me**
- ...

In [ ]:
# ---- setup: the same first cell in every notebook ---------------------------------------

# My own little package. `paths` knows where the tutorial clone, its provided data,
# and my own outputs live, so I never type "../../data/..." by hand.
from lipid_project import paths

paths.ensure_dirs()   # creates data/derived/, figures/, results/ if they're missing (safe to re-run)
paths.summary()       # prints where everything points + which stages I've built so far.
                      # If a line starts with "--", that thing is missing: fix it before going on.

# The usual scientific stack
import numpy as np                 # arrays and math
import pandas as pd                # tables (obs / var are pandas DataFrames)
import matplotlib.pyplot as plt    # plotting
import anndata as ad               # the AnnData container: X (pixels x lipids) + obs + var + obsm + layers

# The tutorial's helper package (installed from my clone in external/).
# Rule from the course: before calling any cl.<module>.<function>, open
# external/lipidomics_tutorial_cajalcourse/src/cajal_lipidomics/<module>.py and read it.
import cajal_lipidomics as cl
cl.style.set_style()               # the course's figure defaults, so plots look clean and consistent

# If this prints without a red error, the kernel is right. A ModuleNotFoundError here almost
# always means the wrong kernel: re-pick "cajal-lipidomics" in the top-right kernel picker.
print("ready. cajal_lipidomics", cl.__version__)

## load

In [ ]:
# ---- load the previous stage --------------------------------------------------------------
# My NB06 output: every pixel now has a lipizone. From here on the pipeline doesn't save new
# stages, NB07 / NB08 / NB09 all read this same file.
adata = ad.read_h5ad(paths.stage("clustered"))

print("pixels x lipids:", adata.shape)
print("conditions:", dict(adata.obs["Condition"].value_counts()))
print("lipizones:", adata.obs["lipizone"].nunique())

# The two rules for this notebook:
#   1. differential tests use layers["umaia"] (real measured intensities), NEVER X_harmony
#   2. fold change direction is fixed: log2(pregnant / control), so positive = up in pregnancy

## my work

In [ ]:
# my code from here on, one small step per cell, and look at the output each time